# VLQ (IQM Quantum Hardware via LEXIS & HEAppE) - Bell State Demo

This notebook demonstrates submitting and executing a quantum circuit (**Bell state**) on the real **VLQ** (IQM) quantum computer using the **LEXIS** test environment and the **QHEAppE** scheduler on the **LUMI** supercomputer.

- **LUMI Project**: `your_lumi_project_id`
- **Repository Branch**: [`40-integration-of-qscheduler---qheappe-client-v100`](https://github.com/It4innovations/quantum-as-a-service/tree/40-integration-of-qscheduler---qheappe-client-v100)
- **LEXIS Project**: `lumiiq`
- **Resource**: `VLQ-CZ`
- **Backend**: `VLQ` (Production)
- **LEXIS API URL**: `https://api.lexis.tech`

### 0. Dependencies Note
When running under the **** environment module, all required packages (, , ) are already pre-installed in the container image. You can skip directly to Step 1.


In [ ]:
# Pre-installed in the lumi-iq environment module. Only run if in a blank non-container environment:
# %pip install --extra-index-url https://opencode.it4i.eu/api/v4/projects/107/packages/pypi/simple matplotlib qaas==0.5.3


### 1. Imports and Interactive Browser Login (OAuth Device Flow)
On LUMI Open OnDemand, you can authenticate **interactively in your browser**:
- If no token is provided, `QProvider` automatically initiates the **OAuth Device Flow**.
- It prints a URL and an authentication code in the cell output.
- You click the link, log in with **MyAccessID / LEXIS**, approve access, and the notebook continues automatically!

In [ ]:
import os
import atexit
import warnings

warnings.filterwarnings("ignore", message=".*IProgress not found.*")


# Clean shutdown handler for Py4HEAppE ThreadPool teardown at Python exit
try:
    from py4heappe.heappe_v6.core.api_client import ApiClient
    _active_heappe_clients = []
    _orig_heappe_init = ApiClient.__init__
    def _hooked_heappe_init(self, *args, **kwargs):
        _orig_heappe_init(self, *args, **kwargs)
        _active_heappe_clients.append(self)
    ApiClient.__init__ = _hooked_heappe_init
    ApiClient.__del__ = lambda self: None
    def _cleanup_heappe_pools():
        for client in _active_heappe_clients:
            try:
                client.pool.close()
                client.pool.terminate()
            except Exception:
                pass
    atexit.register(_cleanup_heappe_pools)
except Exception:
    pass

from qiskit import QuantumCircuit
from qaas.client import QProvider

# 1. Manage LEXIS Token (Environment variable or ~/.lumi-iq/token)
token_dir = os.path.expanduser("~/.lumi-iq")
token_file = os.path.join(token_dir, "token")

token = None
candidate = os.getenv("LEXIS_TOKEN")
if not candidate and os.path.exists(token_file):
    try:
        with open(token_file, "r") as f:
            candidate = f.read().strip()
    except Exception as e:
        print(f"Notice: Could not read {token_file}: {e}")

if candidate:
    try:
        import jwt, time
        payload = jwt.decode(candidate, options={"verify_signature": False})
        exp = payload.get("exp", 0)
        remaining = exp - int(time.time())
        if remaining > 60:
            token = candidate
            print(f"Using valid token (expires in {remaining}s / {remaining // 60}m {remaining % 60}s).")
        else:
            print(f"Token in {token_file} is expired ({remaining}s remaining).")
    except Exception as e:
        print(f"Notice: Could not decode token: {e}")

if not token:
    print("OAuth 2.0 Device Code Flow active.")
    print("-> QProvider will launch the interactive browser login in Step 3.")


### 2. Build the Quantum Circuit (Bell State)
Create a 2-qubit circuit with 3 classical measurement bits:
1. Apply Hadamard gate $H$ on qubit 0
2. Apply CNOT gate ($CX$) from qubit 0 to qubit 1 (creating entangled Bell state $\frac{|00\rangle + |11\rangle}{\sqrt{2}}$)
3. Measure qubits 0 and 1 into classical bits 0 and 1
4. Apply Hadamard gate $H$ on qubit 0 and measure into classical bit 2

In [ ]:
# 1. Build a Qiskit circuit (Bell state)
qc = QuantumCircuit(2, 2)
qc.h(0)
qc.cx(0, 1)
qc.measure_all()

print(qc.draw("text"))


### 3. Initialize QProvider (Interactive Browser Login)
Connecting to the LEXIS testing API (`https://api.lexis.tech`) for project `lumiiq`.

> **Note**: Click the login link and enter the code shown in the cell output below.

In [ ]:
# 2. Instantiate QProvider with your LEXIS project
try:
    from config import LEXIS_PROJECT, LEXIS_RESOURCE
    lexis_project = LEXIS_PROJECT
    resource_name = LEXIS_RESOURCE
except Exception:
    lexis_project = os.getenv("LEXIS_PROJECT", "lumiiq")
    resource_name = os.getenv("LEXIS_RESOURCE", "VLQ-CZ")

print(f"Connecting to LEXIS project: {lexis_project}...")
provider = QProvider(
    lexis_project=lexis_project,
    token=token if token else None,
)

# Securely persist token to ~/.lumi-iq/token with chmod 600
if provider._token:
    try:
        os.makedirs(token_dir, mode=0o700, exist_ok=True)
        fd = os.open(token_file, os.O_WRONLY | os.O_CREAT | os.O_TRUNC, 0o600)
        with open(fd, "w") as f:
            f.write(provider._token)
        os.chmod(token_file, 0o600)
        print(f"Token saved securely to {token_file} (permissions: 0600).")
    except Exception as e:
        print(f"Notice: Could not save token: {e}")

print(f"QProvider successfully authenticated for project: {lexis_project}")


### 4. Initialize Production VLQ Backend and Transpile Circuit
Retrieve the production `VLQ` backend and transpile the circuit to native VLQ (IQM Star topology) basis gates and coupling map.

In [ ]:
# 3. Initialize the QBackend with the client
backend = provider.get_backend(
    lexis_resource=resource_name,
    backend_name="VLQ"  # Production VLQ backend!
)
print(f"Backend: {backend}")

# 4. Transpile the circuit for VLQ Star topology
qc_transpiled = backend.transpile(qc)
print("Circuit transpiled successfully for VLQ:")
print(qc_transpiled.draw("text"))

### 5. Submit Circuit Job
Submit the transpiled circuit to the QScheduler queue (1024 shots).

In [ ]:
# 5. Submit the circuit job
job = backend.run(qc_transpiled, shots=1024, job_name="demo_bell_state")
print(f"Job submitted successfully. Job ID: {job.job_id()}")

### 6. Await Job Execution and Retrieve Results
The call `job.result()` blocks until a terminal state (`FINISHED`, `FAILED`, `CANCELED`) is reached.
We extract results using both **legacy** Qiskit and **primitive** interfaces.

In [ ]:
# 6. Await job execution and retrieve results
print("Waiting for job execution to complete on VLQ...")
result = job.result()
counts = result.get_counts()

print("\n================ Execution Results ================")
print("Counts:", counts)


### 7. Visualize Measurement Results

In [ ]:
try:
    from qiskit.visualization import plot_histogram
    display(plot_histogram(counts, title="VLQ - Bell State Measurement Counts (1024 shots)"))
except Exception as err:
    print("Could not render histogram plot:", err)
    print("Counts:", counts)